# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MrNICEForBonusWinner/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

One of the paper's findings is that the proposed approach improves performance compared to the baseline methods. This is an encouraging result, but I would want to understand exactly how the evaluation was designed. My main question would be whether the validation split truly reflects how the model would be used in practice. If similar examples appear in both the training and testing data, the reported performance could be more optimistic than what would happen on unseen data. Clarifying this helps build confidence that the improvement is genuine.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# Load dataset
df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

# Handle missing values
df = df.fillna(0)

# Save groups before removing client_id
groups = df["client_id"]

# Select numeric features and remove target + identifier
X = df.select_dtypes(include=["number"]).drop(
    columns=["clicks_90d", "client_id"],
    errors="ignore"
)

# Target
y = df["clicks_90d"]

# Honest Group Split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Train model
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")

# Compare with Week 4 baseline
comparison = pd.DataFrame({
    "Approach": ["Week 4 Baseline", "Decision Tree Regressor"],
    "Metric": ["Priority Score", f"MAE = {mae:.2f}"]
})

display(comparison)

Mean Absolute Error (MAE): 2.38


,Approach,Metric
0,Week 4 Baseline,Priority Score
1,Decision Tree Regressor,MAE = 2.38


Another finding is that the model can provide useful recommendations for content optimization. My methodology question focuses on how the labels were created. I would like to know whether the labels were derived from observed user behavior, expert annotations, or rule-based methods. Understanding the label source is important because label quality directly affects model performance and the reliability of its recommendations.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

For this project, I reviewed the features used by my model to make sure they would realistically be available at prediction time. I specifically checked whether any feature directly or indirectly revealed the target value before the prediction was made.

The features I used describe the content and its search performance, while the target variable is the number of clicks. During the audit, I did not intentionally include any feature that directly contains the answer. However, some features, such as impressions and click-through rate (CTR), are closely related to clicks. Because of this relationship, they should be considered carefully to ensure they represent information that would genuinely be available when making a prediction rather than information that depends on future outcomes.

This review helped me better understand that avoiding leakage is not only about removing obvious target columns, but also about thinking carefully about when and how each feature becomes available.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Claim

My model predicts website clicks accurately.

Rewritten Claim

Based on the evaluation performed in this project, the model showed measurable predictive performance on the available dataset. The results suggest that it may be useful as a decision-support tool for understanding content performance, but additional validation on new and unseen data would be needed before making stronger claims about its general performance.

## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.